# General preprocessing pipeline – example call

This call preserves the existing random-forest algorithm. Data always pass through outlier detection, missing-value imputation, and optional standardization in that order.

In [1]:
from pathlib import Path
import importlib
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / 'pyproject.toml').is_file() and (PROJECT_ROOT / 'src').is_dir()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Repository root not found. Start Jupyter from within the repository.')
    PROJECT_ROOT = PROJECT_ROOT.parent
source_root = str(PROJECT_ROOT / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)

import gap_imputation_benchmark.algorithm.rf_domain_imputation as pipeline
pipeline = importlib.reload(pipeline)
DomainImputationConfig = pipeline.DomainImputationConfig
RunMetadata = pipeline.RunMetadata
impute_with_rf_selector = pipeline.impute_with_rf_selector
print('Repository root located.')

Repository root located.


In [2]:
OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'algorithm' / 'synthetic_example' / 'outputs'
OUTPUT_PATH = OUTPUT_DIR / 'synthetic_signal_imputed.csv'  # data_3
PROVENANCE_PATH = OUTPUT_DIR / 'synthetic_signal_provenance.json'

# Executor and responsible person: documentation only.
EXECUTOR_NAME = "Oliver"
EXECUTOR_INFO_PATH = (
    PROJECT_ROOT
    / "data"
    / "person_info_examples"
    / "executor_example.json"
)

RESPONSIBLE_PERSON_NAME = "Jenny"
RESPONSIBLE_PERSON_INFO_PATH = (
    PROJECT_ROOT
    / "data"
    / "person_info_examples"
    / "executor_responsible_person_example.json"
)
EXECUTION_NOTEBOOK_PATH = PROJECT_ROOT / 'notebooks' / 'algorithm' / 'synthetic_example' / '01_example_pipeline_call.ipynb'

# Self-contained example input: a small signal with an artificial gap.
frame = pd.DataFrame({'x': __import__('numpy').sin(__import__('numpy').linspace(0, 10, 300))})
frame.loc[150:151, 'x'] = float('nan')
VALUE_COLUMN = 'x'

In [3]:
config = DomainImputationConfig(
    domain='eye_tracking',
    sampling_rate_hz=1_000.0,
    outlier_method='zscore',
    outlier_threshold=2.1,
    standardization_method='zscore',  # None | zscore | robust | minmax
)
run_metadata = RunMetadata(
    executor_name=EXECUTOR_NAME,
    executor_info_path=EXECUTOR_INFO_PATH,
    executor_responsible_person=RESPONSIBLE_PERSON_NAME,
    executor_responsible_person_info_path=RESPONSIBLE_PERSON_INFO_PATH,
    execution_notebook_path=EXECUTION_NOTEBOOK_PATH,
    comment="Test",
)

imputed_frame, provenance = impute_with_rf_selector(
    frame, VALUE_COLUMN, config=config, run_metadata=run_metadata,
    output_path=OUTPUT_PATH, provenance_path=PROVENANCE_PATH,
)
provenance['summary']

{'gaps_before_outlier_detection': 1,
 'gaps_after_outlier_detection': 1,
 'filled_gap_count': 1,
 'skipped_gap_count': 0}